# Avance Fase 3 — Semana 2: Núcleo Algorítmico, Eficiencia y Programación Orientada a Objetos
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Docente:** Dr. Omar Iván Salinas Silva  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal

---
## 1. Configuración del Entorno, Pregunta Analítica y Carga Segura desde `src/`
- **Pregunta del proyecto:** *¿Qué establecimientos (`id_vu`) se mantienen como generadores de residuos peligrosos entre 2023, 2024 y 2025, cuánto varía su volumen (`cantidad_kg`) año a año, y qué parte del cambio total se explica por establecimientos que entran o salen del registro?*
- **Objetivo general analítico:** *Determinar la persistencia interanual de los establecimientos generadores de residuos peligrosos en Chile durante el trienio 2023–2025 y cuantificar el peso relativo del núcleo persistente frente a la entrada y salida de establecimientos en la variación del volumen total declarado, según región y rubro productivo.*


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Posicionar el directorio de trabajo en la raíz del repositorio para importar el paquete compartido src/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src import (
    COLUMNAS_ESPERADAS_RETC,
    cargar_datos_seguro,
    diagnosticar_variables,
    depurar_registros_base,
    estandarizar_regiones_y_peligrosidad,
    rescatar_coordenadas_chile,
    agregar_atributos_deterministas,
    Transformador,
    ImputadorPorGrupo,
    CodificadorCategorias,
    EscaladorRobusto,
    EstrategiaEliminarFilas,
    EstrategiaMediaGlobal,
    EstrategiaMedianaGlobal,
    EstrategiaMedianaPorGrupo,
    ImputadorFlexible,
    Pipeline,
    BitacoraObserver,
    comparar_implementaciones,
    medir_curva_crecimiento,
    normalizar_peligrosidad_fila_a_fila,
    normalizar_peligrosidad_por_unicos,
    rescatar_serie_coordenada_bucle,
    rescatar_serie_coordenada_vectorizada,
    persistencia_con_bucle,
    persistencia_vectorizada,
    construir_arbol_ciiu,
    sumar_volumen_recursivo,
    aplanar_estructura_recursiva,
)

sns.set_theme(style="whitegrid", palette="deep")
RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")
df_raw = cargar_datos_seguro(RUTA_RAW, columnas_esperadas=COLUMNAS_ESPERADAS_RETC)
print(f"[OK] Raíz del proyecto: {Path.cwd()}")
print(f"[OK] Base Maestra cargada: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")


[OK] Raíz del proyecto: C:\Users\Usuario\proyecto-grupo3-mcdi500
[OK] Base Maestra cargada: 118,280 filas × 21 columnas


## 2. Etapa Funcional Sin Estado (`src/limpieza.py`) y Correcciones de Fase 2
Siguiendo el criterio de diseño visto en clases (*"un componente se convierte en clase solo cuando necesita recordar algo entre una llamada y otra"*), las operaciones deterministas que aplican siempre la misma regla se ejecutan como funciones puras en `src/limpieza.py`:
1. **`depurar_registros_base(df)`:** Elimina las `118` filas duplicadas exactas y descarta las `21` declaraciones sin masa declarada (`cantidad_kg` nulo), ya que `cantidad_kg` es la variable objetivo del análisis y no debe inventarse por imputación. Quedan **`118.141` declaraciones válidas**.
2. **`estandarizar_regiones_y_peligrosidad(df)`:** Homologa `region` a las `16` regiones de Chile y reduce las `812` variantes de `peligrosidad` a `93` categorías canónicas.
3. **`rescatar_coordenadas_chile(df)`:** Recupera de forma vectorizada el punto decimal en `latitud` y `longitud` del año 2024 dentro de los límites de Chile.
4. **`agregar_atributos_deterministas(df)`:** Calcula la persistencia trianual del establecimiento (`n_anios_declarados_vu` y `es_generador_persistente_bin`) sobre el registro completo, el umbral D.S. 148 (`es_gran_generador_bin`) y `anio_indice` (habiendo eliminado la codificación cíclica del año y las variables de largo de texto que carecían de justificación analítica).


In [2]:
df_limpio = depurar_registros_base(df_raw)
df_limpio = estandarizar_regiones_y_peligrosidad(df_limpio, usar_optimizado=True)
df_limpio = rescatar_coordenadas_chile(df_limpio, usar_vectorizado=True)
df_base_f3 = agregar_atributos_deterministas(df_limpio)

print(f"Filas originales: {len(df_raw):,} -> Tras eliminar 118 duplicados y 21 sin masa: {len(df_base_f3):,}")
print(f"Regiones únicas: {df_base_f3['region'].nunique()} | Glosas de peligrosidad canónicas: {df_base_f3['peligrosidad'].nunique()}")
display(df_base_f3[["año", "id_vu", "region", "cantidad_kg", "n_anios_declarados_vu", "es_generador_persistente_bin"]].head())


Filas originales: 118,280 -> Tras eliminar 118 duplicados y 21 sin masa: 118,141
Regiones únicas: 16 | Glosas de peligrosidad canónicas: 93


,año,id_vu,region,cantidad_kg,n_anios_declarados_vu,es_generador_persistente_bin
0,2023,5,Arica y Parinacota,24750.0,3,1
1,2023,6,Arica y Parinacota,15.0,3,1
2,2023,6,Arica y Parinacota,28.0,3,1
3,2023,6,Arica y Parinacota,28.0,3,1
4,2023,6,Arica y Parinacota,80.0,3,1


## 3. Patrón de Diseño `Strategy`: Comparación de las 4 Estrategias de Imputación en un Bucle
En la Fase 2 comparamos cuatro formas de tratar los valores faltantes mediante bloques separados. En la Fase 3 aplicamos el patrón **Strategy**: `ImputadorFlexible(columna, estrategia)` recibe como parámetro cualquiera de las cuatro estrategias (`EstrategiaEliminarFilas`, `EstrategiaMediaGlobal`, `EstrategiaMedianaGlobal`, `EstrategiaMedianaPorGrupo`), permitiendo evaluarlas con un único ciclo `for` sin modificar la clase imputadora.


In [3]:
estrategias_latitud = [
    EstrategiaEliminarFilas(),
    EstrategiaMediaGlobal(),
    EstrategiaMedianaGlobal(),
    EstrategiaMedianaPorGrupo(col_grupo="region"),
]

comparativa_strategy = []
for est in estrategias_latitud:
    imputador_temp = ImputadorFlexible("latitud", estrategia=est)
    df_temp = imputador_temp.ajustar_transformar(df_base_f3)
    s_lat = df_temp["latitud"]
    comparativa_strategy.append({
        "estrategia": est.nombre,
        "filas_resultantes": len(df_temp),
        "pct_filas_perdidas": round(((len(df_base_f3) - len(df_temp)) / len(df_base_f3)) * 100, 2),
        "media_latitud": round(float(s_lat.mean()), 4),
        "mediana_latitud": round(float(s_lat.median()), 4),
        "std_latitud": round(float(s_lat.std()), 4),
    })

df_tabla_strategy = pd.DataFrame(comparativa_strategy)
display(df_tabla_strategy)



,estrategia,filas_resultantes,pct_filas_perdidas,media_latitud,mediana_latitud,std_latitud
0,1. Eliminación por lista (dropna),113125,4.25,-34.7049,-33.6032,5.9160
1,2. Media global,118141,0.00,-34.7287,-33.6098,5.9351
2,3. Mediana global,118141,0.00,-34.7260,-33.6078,5.9354
3,4. Mediana agrupada por 'region' (Elegida),118141,0.00,-34.7305,-33.6079,5.9418


## 4. Núcleo Orientado a Objetos (`Transformador`, Clases Hijas, `Pipeline` y Patrón `Observer`)
Se instancia el `Pipeline` componiendo las tres clases derivadas de `Transformador`:
1. **`ImputadorPorGrupo`:** Aprende las medianas regionales de `latitud` y `longitud` (`279` y `246` nulos tras el rescate geográfico) y la moda por `id_rubro` para `id_lista_a` (`4.740` nulos en 2023–2024) y variables CIIU (`1` nulo), creando las banderas `_imputado`.
2. **`CodificadorCategorias`:** Aprende las Top-10 comunas, transforma `rut_razon_social`, `id_ciiu4` e `id_lista_a` en frecuencias (anonimizando los RUT) y aplica *One-Hot Encoding* sobre `region` (16), `estado_materia` (4) y `comuna_agrupada` (11).
3. **`EscaladorRobusto`:** Aprende mediana y RIC de las variables continuas/discretas y genera las columnas `_rob`, descartando las columnas textuales ya codificadas.

Además, se suscribe `BitacoraObserver` (**Patrón Observer**) para que la bitácora técnica (`F3/docs/bitacora_f3.md`) se genere automáticamente durante la ejecución.


In [4]:
COLS_NUM_IMPUTAR = ["latitud", "longitud"]
COLS_CAT_IMPUTAR = ["id_lista_a", "id_ciiu4", "ciiu4", "id_ciiu6", "ciiu6", "rubro"]
COLS_ESCALAR = ["cantidad_kg", "latitud", "longitud", "frecuencia_declaraciones_rut", "n_peligrosidades"]

observador_bitacora = BitacoraObserver(ruta_salida="F3/docs/bitacora_f3.md")

pipeline_f3 = Pipeline([
    ImputadorPorGrupo(cols_numericas=COLS_NUM_IMPUTAR, cols_categoricas=COLS_CAT_IMPUTAR),
    CodificadorCategorias(top_n_comunas=10),
    EscaladorRobusto(columnas=COLS_ESCALAR, descartar_texto=True),
]).suscribir(observador_bitacora)

df_procesado_f3 = pipeline_f3.ajustar_transformar(df_base_f3)

print(f"[OK] Pipeline POO ejecutado: {df_procesado_f3.shape[0]:,} filas × {df_procesado_f3.shape[1]} columnas (0 nulos)")
print("\nRegistro automático capturado por BitacoraObserver (Patrón Observer):")
display(observador_bitacora.obtener_tabla())



[OK] Pipeline POO ejecutado: 118,141 filas × 61 columnas (0 nulos)

Registro automático capturado por BitacoraObserver (Patrón Observer):


,orden,paso,modo,filas_in,filas_out,cols_in,cols_out,nulos_in,nulos_out,tiempo_s,resumen_parametros
0,1,ImputadorPorGrupo,ajustar,118141,118141,27,35,5270,0,0.3545,2 vars numéricas por región y 6 categóricas po...
1,2,CodificadorCategorias,ajustar,118141,118141,35,70,0,0,0.4776,"Top-10 comunas, 6777 RUTs, 16 regiones, 4 estados"
2,3,EscaladorRobusto,ajustar,118141,118141,70,61,0,0,0.0832,5 variables escaladas con mediana y RIC


## 5. Demostración de Herencia, Encapsulamiento y Separación Entrenamiento / Prueba (Sin Fuga de Datos)
Se comprueba programáticamente:
1. Que las tres clases hijas son instancias tanto de su propia clase como de `Transformador` (`isinstance`), heredando los métodos `ajustar`, `transformar` y la propiedad `parametros`.
2. Que al ajustar el `Pipeline` únicamente con el subconjunto de **entrenamiento (80%)** y aplicarlo con `.transformar()` sobre el subconjunto de **prueba (20%)**, ambos conjuntos conservan exactamente las mismas `56` columnas numéricas y en el mismo orden, evitando toda fuga de datos (*data leakage*).


In [5]:
for paso in pipeline_f3.pasos:
    assert isinstance(paso, Transformador), f"{paso.nombre} no hereda de Transformador"
    print(f"[HERENCIA OK] {paso.nombre} -> es subclase de Transformador | Ajustado: {paso.ajustado}")

# Demostración de ajuste en entrenamiento (80%) y aplicación en prueba (20%) sin fuga de datos
df_train = df_base_f3.sample(frac=0.8, random_state=42)
df_test = df_base_f3.drop(index=df_train.index)

pipe_split = Pipeline([
    ImputadorPorGrupo(cols_numericas=COLS_NUM_IMPUTAR, cols_categoricas=COLS_CAT_IMPUTAR),
    CodificadorCategorias(top_n_comunas=10),
    EscaladorRobusto(columnas=COLS_ESCALAR, descartar_texto=True),
])
pipe_split.ajustar(df_train)
salida_train = pipe_split.transformar(df_train)
salida_test = pipe_split.transformar(df_test)

assert list(salida_train.columns) == list(salida_test.columns), "Las columnas de train y test no coinciden"
print(f"[SIN FUGA DE DATOS OK] Entrenamiento: {salida_train.shape} | Prueba: {salida_test.shape} (esquema idéntico)")


[HERENCIA OK] ImputadorPorGrupo -> es subclase de Transformador | Ajustado: True
[HERENCIA OK] CodificadorCategorias -> es subclase de Transformador | Ajustado: True
[HERENCIA OK] EscaladorRobusto -> es subclase de Transformador | Ajustado: True
[SIN FUGA DE DATOS OK] Entrenamiento: (94513, 61) | Prueba: (23628, 61) (esquema idéntico)


## 6. Verificación de Equivalencia Exacta contra la Fase 2 (`pd.testing.assert_frame_equal`)
Se contrasta la matriz producida por el `Pipeline` orientado a objetos de la Fase 3 contra el archivo procesado de referencia de la Fase 2 (`F2/data/processed/base_maestra_2023_2025_clean.csv`). La coincidencia exacta demuestra que la reorganización del código en clases preservó íntegramente el resultado analítico.


In [6]:
RUTA_CLEAN_F2 = Path("F2/data/processed/base_maestra_2023_2025_clean.csv")
RUTA_CLEAN_F3 = Path("F3/data/processed/base_maestra_2023_2025_clean_f3.csv")

df_referencia_f2 = pd.read_csv(RUTA_CLEAN_F2)
pd.testing.assert_frame_equal(df_referencia_f2, df_procesado_f3, check_exact=False, rtol=1e-5, atol=1e-5)
print("[VERIFICACIÓN DE EQUIVALENCIA EXITOSA] pd.testing.assert_frame_equal pasó sin diferencias:")
print(f" -> Fase 2: {df_referencia_f2.shape} == Fase 3 (POO): {df_procesado_f3.shape}")

df_procesado_f3.to_csv(RUTA_CLEAN_F3, index=False, encoding="utf-8")


[VERIFICACIÓN DE EQUIVALENCIA EXITOSA] pd.testing.assert_frame_equal pasó sin diferencias:
 -> Fase 2: (118141, 61) == Fase 3 (POO): (118141, 61)


## 7. Validación Técnica Sistemática: Casos Normales, Casos Límite y Excepciones
Tal como exige el apartado **II.c**, se verifica el comportamiento de las clases en tres escenarios:
1. **Caso normal:** Ejecución sobre las `118.141` declaraciones reales sin valores nulos remanentes ni pérdida de filas.
2. **Casos límite:** (a) Columna sin valores nulos no se altera; (b) Columna con una sola categoría y varianza cero ($\text{RIC} = 0$) no produce división por cero (`NaN`/`inf`); (c) Categoría nueva en el conjunto de prueba no rompe el esquema de columnas One-Hot.
3. **Excepciones controladas:** (a) Llamar a `.transformar()` antes de `.ajustar()` lanza `RuntimeError`; (b) Pasar una columna inexistente lanza `KeyError`; (c) Pasar una ruta inexistente a `cargar_datos_seguro()` lanza `FileNotFoundError`.


In [7]:
# 1. CASO NORMAL
assert df_procesado_f3.shape == (118141, 10), f"Dimensiones inesperadas: {df_procesado_f3.shape}"
assert int(df_procesado_f3.isna().sum().sum()) == 0, "Quedaron valores nulos en el resultado"
print("[1. CASO NORMAL OK] 118.141 filas 1 60 columnas 100% numéricas y 0 nulos.")

# 2. CASOS LÍMITE (sin nulos, varianza/RIC cero y categoría nueva en prueba)
df_limite = pd.DataFrame({
    "constante": [50.0, 50.0, 50.0, 50.0],
    "sin_nulos": [10.0, 20.0, 30.0, 40.0],
})
esc_limite = EscaladorRobusto(columnas=["constante", "sin_nulos"], descartar_texto=False)
res_limite = esc_limite.ajustar_transformar(df_limite)
assert res_limite["constante_rob"].isna().sum() == 0, "Varianza cero produjo NaN en EscaladorRobusto"
assert (res_limite["constante_rob"] == 0.0).all(), "Columna constante centrada en mediana debe valer 0.0"
print("[2. CASO LÍMITE OK] Columna con RIC=0 y columna sin nulos procesadas sin división por cero.")

# 3. EXCEPCIONES CONTROLADAS
try:
    EscaladorRobusto(["cantidad_kg"]).transformar(df_base_f3)
    raise AssertionError("Debió lanzar RuntimeError al transformar sin ajustar")
except RuntimeError as err:
    print(f"[3a. EXCEPCIÓN OK] RuntimeError capturado correctamente: {err}")

try:
    EscaladorRobusto(["columna_inexistente_xyz"]).ajustar(df_base_f3)
    raise AssertionError("Debió lanzar KeyError ante columna inexistente")
except KeyError as err:
    print(f"[3b. EXCEPCIÓN OK] KeyError capturado correctamente: {err}")

try:
    cargar_datos_seguro("archivo_que_no_existe_999.xlsx")
    raise AssertionError("Debió lanzar FileNotFoundError")
except FileNotFoundError as err:
    print(f"[3c. EXCEPCIÓN OK] FileNotFoundError capturado correctamente.")


AssertionError: Dimensiones inesperadas: (118141, 61)

## 8. Eficiencia y Optimización: Comparación Empírica con `timeit` (Mínimo) y `tracemalloc` (Memoria)
Siguiendo las reglas de validez del apunte de la Fase 3 (repetir y conservar el **mínimo**, medir sobre las **118.141 filas reales** y verificar equivalencia antes de comparar), se evalúan las tres operaciones críticas del proyecto:
1. **Normalización de las 812 glosas de `peligrosidad`:** Aplicar regex/Unicode fila por fila (`118.141` llamadas) frente a normalizar únicamente los `812` valores únicos en un diccionario y proyectarlos con `.map()`.
2. **Rescate geográfico de coordenadas (`latitud`):** Corregir valor por valor con `.apply()` frente a la versión vectorizada sobre arreglos NumPy/pandas.
3. **Cálculo de persistencia (`id_vu`):** Recorrer las filas con un bucle en Python frente a `groupby("id_vu")["año"].transform("nunique")`.


In [ ]:
resultados_bench = [
    comparar_implementaciones(
        "1. Normalización de 812 glosas (peligrosidad)",
        lambda: normalizar_peligrosidad_fila_a_fila(df_limpio["peligrosidad"]),
        lambda: normalizar_peligrosidad_por_unicos(df_limpio["peligrosidad"]),
        etiqueta_a="Fila por fila (.apply)",
        etiqueta_b="Mapeo de 812 únicos (.map)",
        repeticiones=5,
    ),
    comparar_implementaciones(
        "2. Rescate de coordenadas (latitud)",
        lambda: rescatar_serie_coordenada_bucle(df_raw["latitud"], -57.0, -17.0),
        lambda: rescatar_serie_coordenada_vectorizada(df_raw["latitud"], -57.0, -17.0),
        etiqueta_a="Valor por valor (.apply + while)",
        etiqueta_b="Vectorizada en bloque (NumPy/pandas)",
        repeticiones=5,
    ),
    comparar_implementaciones(
        "3. Persistencia por establecimiento (id_vu)",
        lambda: persistencia_con_bucle(df_limpio),
        lambda: persistencia_vectorizada(df_limpio),
        etiqueta_a="Bucle explícito en Python (dict/set)",
        etiqueta_b="Agrupación vectorizada (groupby + nunique)",
        repeticiones=5,
    ),
]

df_benchmarks = pd.DataFrame(resultados_bench)
display(df_benchmarks)

# Curva de crecimiento por tamaño N para la operación de mayor ganancia (glosas de peligrosidad)
df_curva = medir_curva_crecimiento(
    df_limpio,
    fn_a=lambda d: normalizar_peligrosidad_fila_a_fila(d["peligrosidad"]),
    fn_b=lambda d: normalizar_peligrosidad_por_unicos(d["peligrosidad"]),
    tamanos=[1000, 10000, 50000, len(df_limpio)],
)
print("\nEvolución del costo con el tamaño de entrada N (Normalización de glosas):")
display(df_curva)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(df_curva["n_filas"], df_curva["tiempo_a_s"], marker="o", label="Versión A: Fila por fila (.apply)")
axes[0].plot(df_curva["n_filas"], df_curva["tiempo_b_s"], marker="s", label="Versión B: Mapeo de únicos (.map)")
axes[0].set_title("Complejidad Temporal Empírica: Tiempo Mínimo (s) vs N")
axes[0].set_xlabel("Número de filas (N)")
axes[0].set_ylabel("Tiempo mínimo (segundos)")
axes[0].legend()

axes[1].bar(df_benchmarks["operacion"], df_benchmarks["aceleracion_x"], color=["#1d4b63", "#0f766e", "#2563eb"])
axes[1].set_title("Factor de Aceleración (Speedup) en N = 118.141 filas")
axes[1].set_ylabel("Veces más rápida (×)")
axes[1].tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.show()


## 9. Recursividad Genuina: Árbol Jerárquico CIIU (`Rubro → CIIU4 → CIIU6`) y Aplanado de Metadatos
El dataset RETC posee una jerarquía económica natural de tres niveles (`rubro` $\to$ `id_ciiu4` $\to$ `id_ciiu6`). Se aplica `sumar_volumen_recursivo(arbol_ciiu)` para agregar recursivamente la masa total declarada desde las hojas (`id_ciiu6`) hasta la raíz nacional, comprobando que coincide exactamente con la suma tabular de `cantidad_kg`, y se utiliza `aplanar_estructura_recursiva()` para auditar los parámetros aprendidos por el `Pipeline`.


In [ ]:
arbol_ciiu = construir_arbol_ciiu(df_base_f3)
total_recursivo_kg = sumar_volumen_recursivo(arbol_ciiu)
total_tabular_kg = float(df_base_f3["cantidad_kg"].sum())

assert abs(total_recursivo_kg - total_tabular_kg) < 1e-3, "Diferencia entre suma recursiva y suma tabular"
print(f"[RECURSIVIDAD CIIU OK] Rubros en raíz: {len(arbol_ciiu)} | Masa total recursiva: {total_recursivo_kg:,.2f} kg == Suma tabular: {total_tabular_kg:,.2f} kg")

# Auditar los parámetros internos aprendidos por cada paso del Pipeline mediante aplanado recursivo
metadatos_pipeline = {paso.nombre: paso.parametros for paso in pipeline_f3.pasos}
metadatos_planos = aplanar_estructura_recursiva(metadatos_pipeline)
print(f"[APLANADO RECURSIVO OK] Total de parámetros aprendidos y auditables en el Pipeline: {len(metadatos_planos)}")
